In [1]:
import pandas as pd
import numpy as np

In [2]:
desc_df = pd.read_csv(r"C:\Users\pratik p kakade\Desktop\ai clinic\DATASETS\symptom_Description.csv")
prec_df = pd.read_csv(r"C:\Users\pratik p kakade\Desktop\ai clinic\DATASETS\symptom_precaution.csv")
severity_df = pd.read_csv(r"C:\Users\pratik p kakade\Desktop\ai clinic\DATASETS\Symptom-severity.csv")

In [3]:
severity_df["Symptom"]=severity_df["Symptom"].str.replace("_"," ").str.strip()

In [4]:
clinical_kb=pd.merge(desc_df,prec_df,on="Disease",how="inner")

In [5]:
clinical_kb.head()

,Disease,Description,Precaution_1,Precaution_2,Precaution_3,Precaution_4
0,Drug Reaction,An adverse drug reaction (ADR) is an injury ca...,stop irritation,consult nearest hospital,stop taking drug,follow up
1,Malaria,An infectious disease caused by protozoan para...,Consult nearest hospital,avoid oily food,avoid non veg food,keep mosquitos out
2,Allergy,An allergy is an immune system response to a f...,apply calamine,cover area with bandage,NaN,use ice to compress itching
3,Hypothyroidism,"Hypothyroidism, also called underactive thyroi...",reduce stress,exercise,eat healthy,get proper sleep
4,Psoriasis,Psoriasis is a common skin disorder that forms...,wash hands with warm soapy water,stop bleeding using pressure,consult doctor,salt baths


In [6]:
severity_dict=dict(zip(severity_df["Symptom"],severity_df["weight"]))

In [7]:
available_symptoms = severity_df['Symptom'].unique().tolist()

In [8]:
print(f"✅ Phase 1 Reset Complete.")
print(f"Total Verified Diseases: {len(clinical_kb)}")
print(f"Total Tracked Symptoms: {len(available_symptoms)}")

✅ Phase 1 Reset Complete.
Total Verified Diseases: 38
Total Tracked Symptoms: 132


In [9]:
def diagnostic_engine(user_symptoms,clinical_kb,severity_dict):
    results=[]

    user_severity_score=sum([severity_dict.get(s,1) for s in user_symptoms])

    for index,row in clinical_kb.iterrows():
        disease = row["Disease"]
        description =row["Description"]
        precautions = [row['Precaution_1'], row['Precaution_2'], row['Precaution_3'], row['Precaution_4']]

        match_count=0
        for symptom in user_symptoms:
            if symptom.lower() in description.lower():
                match_count+=1

        confidence=(match_count/len(user_symptoms))*100 if user_symptoms else 0

        if confidence>20:
            results.append({
                "Disease":"disease",
                "Confidence": round(confidence, 2),
                "Severity_Score": user_severity_score,
                "Description": description,
                "Precautions": [p for p in precautions if pd.notna(p)]
            })

    return sorted(results, key=lambda x: x['Confidence'], reverse=True)
        

In [10]:
# Example Usage:
symptoms = ['itching', 'skin rash', 'nodal skin eruptions']
predictions = diagnostic_engine(symptoms, clinical_kb, severity_dict)
predictions

[]

In [11]:
# pip install sentence-transformers

In [31]:
from sentence_transformers import SentenceTransformer, util
import torch
import pandas as pd


# 1. Load the Model (Lightweight & Open Source)
model = SentenceTransformer('all-MiniLM-L6-v2')

# 2. Prepare the Knowledge Base
# clinical_kb contains your 41 diseases and descriptions
descriptions = clinical_kb['Description'].tolist()
disease_names = clinical_kb['Disease'].tolist()

# 3. Pre-compute Embeddings for the 41 Descriptions
# We do this once so the search is lightning fast later
description_embeddings = model.encode(descriptions, convert_to_tensor=True)

def semantic_diagnostic_engine(user_input_text, top_k=3):
    """
    Compares user text to disease descriptions using Cosine Similarity.
    """
    # Convert user input (e.g., "itching and red skin") into a vector
    user_embedding = model.encode(user_input_text, convert_to_tensor=True)
    
    # Calculate Cosine Similarity against all 41 diseases
    cosine_scores = util.cos_sim(user_embedding, description_embeddings)[0]
    
    # Get the top matches
    top_results = torch.topk(cosine_scores, k=top_k)
    
    predictions = []
    for score, idx in zip(top_results.values, top_results.indices):
        disease = disease_names[idx]
        
        # Fetch the original data for the report
        row = clinical_kb[clinical_kb['Disease'] == disease].iloc[0]
        
        predictions.append({
            "Disease": disease,
            "Match_Score": round(float(score), 4),
            "Description": row['Description'],
            "Precautions": [row['Precaution_1'], row['Precaution_2'], row['Precaution_3'], row['Precaution_4']]
        })
        
    return predictions



Loading weights: 100%|█████████████████████████████████████████████████████████████| 103/103 [00:00<00:00, 6467.07it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [13]:
# --- TEST RUN ---
user_query = "my skin is very itchy and has red bumpy patches"
results = semantic_diagnostic_engine(user_query)

In [14]:
results

[{'Disease': 'Psoriasis',
  'Match_Score': 0.4713,
  'Description': "Psoriasis is a common skin disorder that forms thick, red, bumpy patches covered with silvery scales. They can pop up anywhere, but most appear on the scalp, elbows, knees, and lower back. Psoriasis can't be passed from person to person. It does sometimes happen in members of the same family.",
  'Precautions': ['wash hands with warm soapy water',
   'stop bleeding using pressure',
   'consult doctor',
   'salt baths']},
 {'Disease': 'Impetigo',
  'Match_Score': 0.3694,
  'Description': "Impetigo (im-puh-TIE-go) is a common and highly contagious skin infection that mainly affects infants and children. Impetigo usually appears as red sores on the face, especially around a child's nose and mouth, and on hands and feet. The sores burst and develop honey-colored crusts.",
  'Precautions': ['soak affected area in warm water',
   'use antibiotics',
   'remove scabs with wet compressed cloth',
   'consult doctor']},
 {'Disea

In [15]:
# The template we will send to Groq
PROMPT_TEMPLATE = """
System: You are a Senior Clinical Decision Support Assistant. 
Context: A patient has reported the following symptoms: {user_input}
The AI model has identified the most likely condition as: {disease_name}
Medical Description: {description}
Precautions: {precautions}
Severity Score: {severity_score}/100

Task: Generate a professional, concise summary for a healthcare provider. 
- Highlight why the symptoms match the description.
- Note the urgency based on the severity score.
- Structure the advice into "Immediate Actions" and "Further Diagnostics".
- Tone: Clinical, objective, and non-alarmist.
"""

In [16]:
from fastapi import FastAPI
from pydantic import BaseModel
from groq import Groq
import os

# Initialize FastAPI and Groq Client
app = FastAPI(title="AI Clinical Support System")
client = Groq(api_key="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") # Use your free Groq key

class SymptomRequest(BaseModel):
    symptoms: str

@app.post("/diagnose")
async def get_diagnosis(request: SymptomRequest):
    # 1. Run Semantic Search (from Phase 2)
    top_matches = semantic_diagnostic_engine(request.symptoms)
    primary_match = top_matches[0]
    
    # 2. Calculate Severity (Logic from Phase 1)
    # severity_score = calculate_total_severity(request.symptoms)
    
    # 3. Call Groq for Clinical Summary
    chat_completion = client.chat.completions.create(
        messages=[
            {"role": "system", "content": "You are a clinical assistant."},
            {"role": "user", "content": PROMPT_TEMPLATE.format(
                user_input=request.symptoms,
                disease_name=primary_match['Disease'],
                description=primary_match['Description'],
                precautions=", ".join(primary_match['Precautions']),
                severity_score=75 # Placeholder for logic
            )}
        ],
        model="llama-3.3-70b-versatile", # High speed, low latency
    )
    
    # 4. Final Response Object
    return {
        "diagnosis": primary_match['Disease'],
        "confidence": primary_match['Match_Score'],
        "clinical_summary": chat_completion.choices[0].message.content,
        "raw_data": {
            "description": primary_match['Description'],
            "precautions": primary_match['Precautions']
        }
    }

In [24]:
%%writefile app.py

import streamlit as st
import pandas as pd
import requests

# Set Page Config
st.set_page_config(page_title="AI Clinical Decision Support", layout="wide")

st.title("🩺 Clinical Decision Support System")
st.markdown("---")

# Sidebar for Patient Demographics (Added Context)
with st.sidebar:
    st.header("Patient Context")
    age = st.number_input("Age", 0, 120, 25)
    gender = st.selectbox("Gender", ["Male", "Female", "Other"])
    st.info("Basic medical history integration ready.")

# Main Layout
col1, col2 = st.columns([1, 1])

with col1:
    st.subheader("Input Symptoms")
    user_input = st.text_area("Describe the patient's symptoms (e.g., 'persistent cough and fever')", height=150)
    
    if st.button("Generate Diagnosis"):
        if user_input:
            with st.spinner("Analyzing Clinical Knowledge Base..."):
                # 1. Logic Call (Connecting to your Phase 2/3 code)
                # For this example, we assume the logic is imported or called via API
                results = semantic_diagnostic_engine(user_input) 
                
                # Store results in session state
                st.session_state['results'] = results
        else:
            st.warning("Please enter symptoms first.")

with col2:
    st.subheader("Diagnostic Insights")
    if 'results' in st.session_state:
        primary = st.session_state['results'][0]
        
        # Confidence Meter
        st.metric(label="Primary Diagnosis", value=primary['Disease'])
        st.progress(primary['Match_Score'])
        st.caption(f"Confidence Score: {int(primary['Match_Score']*100)}%")
        
        # Severity Flag (Using your Severity CSV logic)
        st.error("Urgency Level: HIGH") # Example logic
        
        # Precautions (From your Dataset #2)
        with st.expander("Recommended Precautions"):
            for p in primary['Precautions']:
                st.write(f"• {p}")

# Final Report (The Groq Output)
st.markdown("---")
if 'results' in st.session_state:
    st.subheader("Generated Clinical Summary (LLM)")
    # This would be the response.json()['clinical_summary'] from Step 3
    st.info("Patient presents with indicators of " + primary['Disease'] + ". Further diagnostics required.")

Writing app.py


In [25]:
# pip install streamlit

In [26]:
# Final Logic for the "Insights" Panel
def get_final_report(top_results):
    report = []
    for res in top_results:
        # Normalize scores to 0-100% for the doctor
        confidence_pct = f"{int(res['Match_Score'] * 100)}%"
        report.append(f"Condition: {res['Disease']} | Confidence: {confidence_pct}")
    return report

In [27]:
!streamlit run app.py

^C
